In [39]:
!uv pip install requests pandas pyarrow python-dotenv jupyter ipykernel


Using Python 3.11.3 environment at: /Users/jacodon/Dev/data-platform/.venv
Checked 6 packages in 200ms


requests — fa le chiamate HTTP all'API di Alpha Vantage. È la libreria standard per fare richieste web in Python.  
pandas — manipola i dati in formato tabellare (DataFrame). Lo useremo per normalizzare il JSON dell'API e prepararlo per la conversione in Parquet.  
pyarrow — gestisce il formato Parquet. Pandas lo usa internamente per leggere e scrivere file .parquet.  
python-dotenv — carica il .env in memoria come abbiamo discusso.  
jupyter — il core di Jupyter, necessario per eseguire i notebook.  
ipykernel — permette al .venv di essere riconosciuto come kernel da VS Code. Senza di esso VS Code non saprebbe come collegare il notebook al venv.  

In [ ]:
import sys # per modificare il path di ricerca dei moduli
import os # per accedere alle variabili d'ambiente e al file system
import requests # per fare richieste HTTP all'API di Alpha Vantage
import pandas as pd # per manipolare i dati in formato tabellare
from dotenv import load_dotenv  # per caricare le variabili d'ambiente da un file .env

sys.path.append('.')   # aggiunge ingestion/ al path di ricerca

load_dotenv(dotenv_path='../.env')  # carica le variabili d'ambiente dal file .env
API_KEY = os.getenv('ALPHA_VANTAGE_API_KEY') # recupera la chiave API dalle variabili d'ambiente
print(API_KEY) # per verificare che la chiave API sia stata caricata correttamente


TIME_SERIES_DAILY

In [41]:
ticker = "AAPL" # simbolo del titolo da analizzare, ad esempio Apple

In [42]:
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": ticker,
    "outputsize": "compact",  # ultimi 100 giorni
    "apikey": API_KEY
}

response = requests.get("https://www.alphavantage.co/query", params=params)
print(response.url)

data = response.json()
print(type(data))  # per verificare che la risposta sia un dizionario
print(data.keys())  # per vedere le chiavi principali del dizionario

https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=AAPL&outputsize=compact&apikey=CPQV7Y207E1YCQNB
<class 'dict'>
dict_keys(['Meta Data', 'Time Series (Daily)'])


In [43]:
print(data['Meta Data'])

{'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'AAPL', '3. Last Refreshed': '2026-06-08', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}


In [44]:
print(data['Time Series (Daily)'])

{'2026-06-08': {'1. open': '308.7390', '2. high': '317.4000', '3. low': '301.1700', '4. close': '301.5400', '5. volume': '77949082'}, '2026-06-05': {'1. open': '312.8600', '2. high': '315.1700', '3. low': '307.1500', '4. close': '307.3400', '5. volume': '65310502'}, '2026-06-04': {'1. open': '313.2300', '2. high': '313.5400', '3. low': '309.6500', '4. close': '311.2300', '5. volume': '44869134'}, '2026-06-03': {'1. open': '314.1750', '2. high': '316.9400', '3. low': '308.8500', '4. close': '310.2600', '5. volume': '50836705'}, '2026-06-02': {'1. open': '307.4600', '2. high': '315.4500', '3. low': '306.6850', '4. close': '315.2000', '5. volume': '44534716'}, '2026-06-01': {'1. open': '309.6250', '2. high': '310.9400', '3. low': '305.0200', '4. close': '306.3100', '5. volume': '48849933'}, '2026-05-29': {'1. open': '311.7750', '2. high': '315.0000', '3. low': '309.5300', '4. close': '312.0600', '5. volume': '70026752'}, '2026-05-28': {'1. open': '310.6800', '2. high': '312.8000', '3. low

In [45]:
import json
print(json.dumps(data, indent=2))

{
  "Meta Data": {
    "1. Information": "Daily Prices (open, high, low, close) and Volumes",
    "2. Symbol": "AAPL",
    "3. Last Refreshed": "2026-06-08",
    "4. Output Size": "Compact",
    "5. Time Zone": "US/Eastern"
  },
  "Time Series (Daily)": {
    "2026-06-08": {
      "1. open": "308.7390",
      "2. high": "317.4000",
      "3. low": "301.1700",
      "4. close": "301.5400",
      "5. volume": "77949082"
    },
    "2026-06-05": {
      "1. open": "312.8600",
      "2. high": "315.1700",
      "3. low": "307.1500",
      "4. close": "307.3400",
      "5. volume": "65310502"
    },
    "2026-06-04": {
      "1. open": "313.2300",
      "2. high": "313.5400",
      "3. low": "309.6500",
      "4. close": "311.2300",
      "5. volume": "44869134"
    },
    "2026-06-03": {
      "1. open": "314.1750",
      "2. high": "316.9400",
      "3. low": "308.8500",
      "4. close": "310.2600",
      "5. volume": "50836705"
    },
    "2026-06-02": {
      "1. open": "307.4600",
   

In [ ]:
# Facciamo alcuni test per capire meglio la struttura dei dati

time_series = data['Time Series (Daily)'] # è un dizionario dove le chiavi sono le date e i valori sono altri dizionari con i prezzi
print(time_series) 
print(type(time_series)) # dovrebbe essere un dizionario
print(len(time_series)) # dovrebbe essere 100 per la modalità "compact"

{'2026-06-08': {'1. open': '308.7390', '2. high': '317.4000', '3. low': '301.1700', '4. close': '301.5400', '5. volume': '77949082'}, '2026-06-05': {'1. open': '312.8600', '2. high': '315.1700', '3. low': '307.1500', '4. close': '307.3400', '5. volume': '65310502'}, '2026-06-04': {'1. open': '313.2300', '2. high': '313.5400', '3. low': '309.6500', '4. close': '311.2300', '5. volume': '44869134'}, '2026-06-03': {'1. open': '314.1750', '2. high': '316.9400', '3. low': '308.8500', '4. close': '310.2600', '5. volume': '50836705'}, '2026-06-02': {'1. open': '307.4600', '2. high': '315.4500', '3. low': '306.6850', '4. close': '315.2000', '5. volume': '44534716'}, '2026-06-01': {'1. open': '309.6250', '2. high': '310.9400', '3. low': '305.0200', '4. close': '306.3100', '5. volume': '48849933'}, '2026-05-29': {'1. open': '311.7750', '2. high': '315.0000', '3. low': '309.5300', '4. close': '312.0600', '5. volume': '70026752'}, '2026-05-28': {'1. open': '310.6800', '2. high': '312.8000', '3. low

In [47]:
# estraiamo il primo elemento per vedere la struttura

first_date = list(time_series.keys())[0] # la prima data dovrebbe essere la più recente, ma è meglio verificarlo
print(f"Data più recente: {first_date}") 
print(time_series[first_date])

Data più recente: 2026-06-08
{'1. open': '308.7390', '2. high': '317.4000', '3. low': '301.1700', '4. close': '301.5400', '5. volume': '77949082'}


In [48]:
df = pd.DataFrame.from_dict(time_series, orient='index') # orient='index' perché le chiavi del dizionario (le date) devono diventare gli indici del DataFrame
print(df.head())
print(df.dtypes)

             1. open   2. high    3. low  4. close 5. volume
2026-06-08  308.7390  317.4000  301.1700  301.5400  77949082
2026-06-05  312.8600  315.1700  307.1500  307.3400  65310502
2026-06-04  313.2300  313.5400  309.6500  311.2300  44869134
2026-06-03  314.1750  316.9400  308.8500  310.2600  50836705
2026-06-02  307.4600  315.4500  306.6850  315.2000  44534716
1. open      str
2. high      str
3. low       str
4. close     str
5. volume    str
dtype: object


In [49]:
# rinomina le colonne
df.columns = ['open', 'high', 'low', 'close', 'volume'] # attributo dell'oggetto DataFrame che permette di rinominare le colonne

# converti i tipi
df[['open', 'high', 'low', 'close']] = df[['open', 'high', 'low', 'close']].astype(float) # metodo astype() per convertire i tipi di dati, in questo caso da stringa a float
df['volume'] = df['volume'].astype(int) # il volume è un numero intero, quindi lo convertiamo in int

# converti l'indice in datetime
df.index = pd.to_datetime(df.index) # metodo to_datetime() per convertire le date in formato datetime
df.index.name = 'date' # rinomina l'indice in 'date' per chiarezza

# aggiungi il ticker
df['ticker'] = ticker # aggiungiamo una colonna 'ticker' con il ticker  del titolo

print(df.head()) 
print(df.dtypes)

               open    high      low   close    volume ticker
date                                                         
2026-06-08  308.739  317.40  301.170  301.54  77949082   AAPL
2026-06-05  312.860  315.17  307.150  307.34  65310502   AAPL
2026-06-04  313.230  313.54  309.650  311.23  44869134   AAPL
2026-06-03  314.175  316.94  308.850  310.26  50836705   AAPL
2026-06-02  307.460  315.45  306.685  315.20  44534716   AAPL
open      float64
high      float64
low       float64
close     float64
volume      int64
ticker        str
dtype: object


In [ ]:
df = df.reset_index() # metodo reset_index() per spostare l'indice 'date' come colonna normale
print(df.head())
print(df.dtypes)

        date     open    high      low   close    volume ticker
0 2026-06-08  308.739  317.40  301.170  301.54  77949082   AAPL
1 2026-06-05  312.860  315.17  307.150  307.34  65310502   AAPL
2 2026-06-04  313.230  313.54  309.650  311.23  44869134   AAPL
3 2026-06-03  314.175  316.94  308.850  310.26  50836705   AAPL
4 2026-06-02  307.460  315.45  306.685  315.20  44534716   AAPL
date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker               str
dtype: object


In [62]:
import pyarrow as pa # per lavorare con i dati in formato Parquet
import pyarrow.parquet as pq # per salvare i dati in formato Parquet

output_path = f"staging/prices/{ticker}_prices.parquet" # percorso di output per il file Parquet, con il nome del ticker per identificare il titolo

os.makedirs(f"staging/prices", exist_ok=True) # crea la cartella di destinazione se non esiste già

df.to_parquet(output_path, index=False) # metodo to_parquet() per salvare il DataFrame in formato Parquet, index=False per non salvare l'indice come colonna
print(f"File salvato in: {output_path}") 

df_check = pd.read_parquet(output_path)
print(df_check.head())
print(df_check.dtypes)

File salvato in: staging/prices/AAPL_prices.parquet
        date     open    high      low   close    volume ticker
0 2026-06-08  308.739  317.40  301.170  301.54  77949082   AAPL
1 2026-06-05  312.860  315.17  307.150  307.34  65310502   AAPL
2 2026-06-04  313.230  313.54  309.650  311.23  44869134   AAPL
3 2026-06-03  314.175  316.94  308.850  310.26  50836705   AAPL
4 2026-06-02  307.460  315.45  306.685  315.20  44534716   AAPL
date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker               str
dtype: object


Abbiamo completato l'esplorazione dell'endpoint TIME_SERIES_DAILY:  
✅ chiamata API     
✅ struttura JSON capita        
✅ trasformazione in DataFrame       
✅ tipi corretti        
✅ salvataggio in Parquet verificato        

OVERVIEW

In [56]:
params_overview = {
    "function": "OVERVIEW",
    "symbol": ticker,
    "apikey": API_KEY
}

response_overview = requests.get("https://www.alphavantage.co/query", params=params_overview)
data_overview = response_overview.json()

print(data_overview.keys())
print(data_overview)

dict_keys(['Symbol', 'AssetType', 'Name', 'Description', 'CIK', 'Exchange', 'Currency', 'Country', 'Sector', 'Industry', 'Address', 'OfficialSite', 'FiscalYearEnd', 'LatestQuarter', 'MarketCapitalization', 'EBITDA', 'PERatio', 'PEGRatio', 'BookValue', 'DividendPerShare', 'DividendYield', 'EPS', 'RevenuePerShareTTM', 'ProfitMargin', 'OperatingMarginTTM', 'ReturnOnAssetsTTM', 'ReturnOnEquityTTM', 'RevenueTTM', 'GrossProfitTTM', 'DilutedEPSTTM', 'QuarterlyEarningsGrowthYOY', 'QuarterlyRevenueGrowthYOY', 'AnalystTargetPrice', 'AnalystRatingStrongBuy', 'AnalystRatingBuy', 'AnalystRatingHold', 'AnalystRatingSell', 'AnalystRatingStrongSell', 'TrailingPE', 'ForwardPE', 'PriceToSalesRatioTTM', 'PriceToBookRatio', 'EVToRevenue', 'EVToEBITDA', 'Beta', '52WeekHigh', '52WeekLow', '50DayMovingAverage', '200DayMovingAverage', 'SharesOutstanding', 'SharesFloat', 'PercentInsiders', 'PercentInstitutions', 'DividendDate', 'ExDividendDate'])
{'Symbol': 'AAPL', 'AssetType': 'Common Stock', 'Name': 'Apple I

In [60]:
print(json.dumps(data_overview, indent=2)) # json.dumps() per stampare il dizionario in modo leggibile, con indentazione di 2 spazi

{
  "Symbol": "AAPL",
  "AssetType": "Common Stock",
  "Name": "Apple Inc",
  "Description": "Apple Inc. is an American multinational technology company that specializes in consumer electronics, computer software, and online services. Apple is the world's largest technology company by revenue (totalling $274.5 billion in 2020) and, since January 2021, the world's most valuable company. As of 2021, Apple is the world's fourth-largest PC vendor by unit sales, and fourth-largest smartphone manufacturer. It is one of the Big Five American information technology companies, along with Amazon, Google, Microsoft, and Facebook.",
  "CIK": "320193",
  "Exchange": "NASDAQ",
  "Currency": "USD",
  "Country": "USA",
  "Sector": "TECHNOLOGY",
  "Industry": "CONSUMER ELECTRONICS",
  "Address": "ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STATES, 95014",
  "OfficialSite": "https://www.apple.com",
  "FiscalYearEnd": "September",
  "LatestQuarter": "2026-03-31",
  "MarketCapitalization": "4428825362000",


In [69]:
df_overview = pd.DataFrame([data_overview])
print(df_overview.shape)
print(df_overview.head())
print(df_overview.dtypes)

(1, 55)
  Symbol     AssetType       Name  \
0   AAPL  Common Stock  Apple Inc   

                                         Description     CIK Exchange  \
0  Apple Inc. is an American multinational techno...  320193   NASDAQ   

  Currency Country      Sector              Industry  ... 52WeekHigh  \
0      USD     USA  TECHNOLOGY  CONSUMER ELECTRONICS  ...      317.4   

  52WeekLow 50DayMovingAverage 200DayMovingAverage SharesOutstanding  \
0     194.3             282.21              265.57       14687356000   

   SharesFloat PercentInsiders PercentInstitutions DividendDate ExDividendDate  
0  14662534000           1.632              65.821   2026-05-14     2026-05-11  

[1 rows x 55 columns]
Symbol                        str
AssetType                     str
Name                          str
Description                   str
CIK                           str
Exchange                      str
Currency                      str
Country                       str
Sector                 

In [70]:
cols = [
    'Symbol', 'Name', 'Exchange', 'Currency', 'Country',
    'Sector', 'Industry', 'MarketCapitalization', 'EBITDA',
    'PERatio', 'EPS', 'Beta', '52WeekHigh', '52WeekLow',
    'DividendYield', 'ProfitMargin'
] # selezioniamo solo le colonne che ci interessano per dim company 

df_overview_clean = df_overview[cols].copy()

numeric_cols = [
    'MarketCapitalization', 'EBITDA', 'PERatio', 'EPS',
    'Beta', '52WeekHigh', '52WeekLow', 'DividendYield', 'ProfitMargin'
] # colonne che vogliamo convertire da stringa a numerico

df_overview_clean[numeric_cols] = df_overview_clean[numeric_cols].apply(pd.to_numeric, errors='coerce') # metodo apply() per applicare la funzione pd.to_numeric a tutte le colonne specificate, errors='coerce' per convertire i valori non convertibili in NaN

print(df_overview_clean.dtypes)
print(df_overview_clean.head())

Symbol                      str
Name                        str
Exchange                    str
Currency                    str
Country                     str
Sector                      str
Industry                    str
MarketCapitalization      int64
EBITDA                    int64
PERatio                 float64
EPS                     float64
Beta                    float64
52WeekHigh              float64
52WeekLow               float64
DividendYield           float64
ProfitMargin            float64
dtype: object
  Symbol       Name Exchange Currency Country      Sector  \
0   AAPL  Apple Inc   NASDAQ      USD     USA  TECHNOLOGY   

               Industry  MarketCapitalization        EBITDA  PERatio   EPS  \
0  CONSUMER ELECTRONICS         4428825362000  159975997000    36.51  8.26   

    Beta  52WeekHigh  52WeekLow  DividendYield  ProfitMargin  
0  1.086       317.4      194.3         0.0034         0.272  


In [71]:
df_overview_clean['ticker'] = ticker # aggiungiamo una colonna 'ticker' con il ticker  del titolo

output_path_overview = f"staging/overview/{ticker}_overview.parquet"
os.makedirs("staging/overview", exist_ok=True)

df_overview_clean.to_parquet(output_path_overview, index=False)
print(f"File salvato in: {output_path_overview}")

df_check_overview = pd.read_parquet(output_path_overview)
print(df_check_overview.head())
print(df_check_overview.dtypes)

File salvato in: staging/overview/AAPL_overview.parquet
  Symbol       Name Exchange Currency Country      Sector  \
0   AAPL  Apple Inc   NASDAQ      USD     USA  TECHNOLOGY   

               Industry  MarketCapitalization        EBITDA  PERatio   EPS  \
0  CONSUMER ELECTRONICS         4428825362000  159975997000    36.51  8.26   

    Beta  52WeekHigh  52WeekLow  DividendYield  ProfitMargin ticker  
0  1.086       317.4      194.3         0.0034         0.272   AAPL  
Symbol                      str
Name                        str
Exchange                    str
Currency                    str
Country                     str
Sector                      str
Industry                    str
MarketCapitalization      int64
EBITDA                    int64
PERatio                 float64
EPS                     float64
Beta                    float64
52WeekHigh              float64
52WeekLow               float64
DividendYield           float64
ProfitMargin            float64
ticker      

Abbiamo completato l'esplorazione dell'endpoint OVERVIEW:  
✅ chiamata API     
✅ struttura JSON capita        
✅ trasformazione in DataFrame       
✅ tipi corretti        
✅ salvataggio in Parquet verificato  

EARNINGS

In [73]:
params_earnings = {
    "function": "EARNINGS",
    "symbol": ticker,
    "apikey": API_KEY
}

response_earnings = requests.get("https://www.alphavantage.co/query", params=params_earnings)
data_earnings = response_earnings.json()

print(data_earnings.keys())
print(data_earnings)

dict_keys(['symbol', 'annualEarnings', 'quarterlyEarnings'])
{'symbol': 'AAPL', 'annualEarnings': [{'fiscalDateEnding': '2026-03-31', 'reportedEPS': '4.85'}, {'fiscalDateEnding': '2025-09-30', 'reportedEPS': '7.47'}, {'fiscalDateEnding': '2024-09-30', 'reportedEPS': '6.08'}, {'fiscalDateEnding': '2023-09-30', 'reportedEPS': '6.12'}, {'fiscalDateEnding': '2022-09-30', 'reportedEPS': '6.11'}, {'fiscalDateEnding': '2021-09-30', 'reportedEPS': '5.62'}, {'fiscalDateEnding': '2020-09-30', 'reportedEPS': '3.27'}, {'fiscalDateEnding': '2019-09-30', 'reportedEPS': '2.98'}, {'fiscalDateEnding': '2018-09-30', 'reportedEPS': '2.97'}, {'fiscalDateEnding': '2017-09-30', 'reportedEPS': '2.3'}, {'fiscalDateEnding': '2016-09-30', 'reportedEPS': '2.0675'}, {'fiscalDateEnding': '2015-09-30', 'reportedEPS': '2.3'}, {'fiscalDateEnding': '2014-09-30', 'reportedEPS': '1.6075'}, {'fiscalDateEnding': '2013-09-30', 'reportedEPS': '1.415'}, {'fiscalDateEnding': '2012-09-30', 'reportedEPS': '1.5775'}, {'fiscalDat

In [74]:
print(json.dumps(data_earnings, indent=2)) # json.dumps() per stampare il dizionario in modo leggibile, con indentazione di 2 spazi

{
  "symbol": "AAPL",
  "annualEarnings": [
    {
      "fiscalDateEnding": "2026-03-31",
      "reportedEPS": "4.85"
    },
    {
      "fiscalDateEnding": "2025-09-30",
      "reportedEPS": "7.47"
    },
    {
      "fiscalDateEnding": "2024-09-30",
      "reportedEPS": "6.08"
    },
    {
      "fiscalDateEnding": "2023-09-30",
      "reportedEPS": "6.12"
    },
    {
      "fiscalDateEnding": "2022-09-30",
      "reportedEPS": "6.11"
    },
    {
      "fiscalDateEnding": "2021-09-30",
      "reportedEPS": "5.62"
    },
    {
      "fiscalDateEnding": "2020-09-30",
      "reportedEPS": "3.27"
    },
    {
      "fiscalDateEnding": "2019-09-30",
      "reportedEPS": "2.98"
    },
    {
      "fiscalDateEnding": "2018-09-30",
      "reportedEPS": "2.97"
    },
    {
      "fiscalDateEnding": "2017-09-30",
      "reportedEPS": "2.3"
    },
    {
      "fiscalDateEnding": "2016-09-30",
      "reportedEPS": "2.0675"
    },
    {
      "fiscalDateEnding": "2015-09-30",
      "reportedEPS

In [75]:
print(f"Annual earnings records: {len(data_earnings['annualEarnings'])}")
print(data_earnings['annualEarnings'][0])

print(f"\nQuarterly earnings records: {len(data_earnings['quarterlyEarnings'])}")
print(data_earnings['quarterlyEarnings'][0])

Annual earnings records: 31
{'fiscalDateEnding': '2026-03-31', 'reportedEPS': '4.85'}

Quarterly earnings records: 121
{'fiscalDateEnding': '2026-03-31', 'reportedDate': '2026-04-30', 'reportedEPS': '2.01', 'estimatedEPS': '1.94', 'surprise': '0.07', 'surprisePercentage': '3.6082', 'reportTime': 'post-market'}


In [76]:
df_earnings = pd.DataFrame(data_earnings['quarterlyEarnings'])
df_earnings['ticker'] = ticker

print(df_earnings.shape)
print(df_earnings.dtypes)
print(df_earnings.head())

(121, 8)
fiscalDateEnding      str
reportedDate          str
reportedEPS           str
estimatedEPS          str
surprise              str
surprisePercentage    str
reportTime            str
ticker                str
dtype: object
  fiscalDateEnding reportedDate reportedEPS estimatedEPS surprise  \
0       2026-03-31   2026-04-30        2.01         1.94     0.07   
1       2025-12-31   2026-01-29        2.84         2.67     0.17   
2       2025-09-30   2025-10-30        1.85         1.77     0.08   
3       2025-06-30   2025-07-31        1.57         1.43     0.14   
4       2025-03-31   2025-05-01        1.65         1.62     0.03   

  surprisePercentage   reportTime ticker  
0             3.6082  post-market   AAPL  
1              6.367  post-market   AAPL  
2             4.5198  post-market   AAPL  
3             9.7902  post-market   AAPL  
4             1.8519  post-market   AAPL  


In [77]:
# converti le date
df_earnings['fiscalDateEnding'] = pd.to_datetime(df_earnings['fiscalDateEnding'])
df_earnings['reportedDate'] = pd.to_datetime(df_earnings['reportedDate'])

# converti i float
float_cols = ['reportedEPS', 'estimatedEPS', 'surprise', 'surprisePercentage']
df_earnings[float_cols] = df_earnings[float_cols].apply(pd.to_numeric, errors='coerce')

print(df_earnings.dtypes)

fiscalDateEnding      datetime64[us]
reportedDate          datetime64[us]
reportedEPS                  float64
estimatedEPS                 float64
surprise                     float64
surprisePercentage           float64
reportTime                       str
ticker                           str
dtype: object


In [78]:
# salva in Parquet
output_path_earnings = f"staging/earnings/{ticker}_earnings.parquet"
os.makedirs("staging/earnings", exist_ok=True)

df_earnings.to_parquet(output_path_earnings, index=False)
print(f"File salvato in: {output_path_earnings}")

# verifica
df_check_earnings = pd.read_parquet(output_path_earnings)
print(df_check_earnings.head())
print(df_check_earnings.dtypes)

File salvato in: staging/earnings/AAPL_earnings.parquet
  fiscalDateEnding reportedDate  reportedEPS  estimatedEPS  surprise  \
0       2026-03-31   2026-04-30         2.01          1.94      0.07   
1       2025-12-31   2026-01-29         2.84          2.67      0.17   
2       2025-09-30   2025-10-30         1.85          1.77      0.08   
3       2025-06-30   2025-07-31         1.57          1.43      0.14   
4       2025-03-31   2025-05-01         1.65          1.62      0.03   

   surprisePercentage   reportTime ticker  
0              3.6082  post-market   AAPL  
1              6.3670  post-market   AAPL  
2              4.5198  post-market   AAPL  
3              9.7902  post-market   AAPL  
4              1.8519  post-market   AAPL  
fiscalDateEnding      datetime64[us]
reportedDate          datetime64[us]
reportedEPS                  float64
estimatedEPS                 float64
surprise                     float64
surprisePercentage           float64
reportTime               

Abbiamo completato l'esplorazione dell'endpoint EARNINGS:  
✅ chiamata API     
✅ struttura JSON capita        
✅ trasformazione in DataFrame       
✅ tipi corretti        
✅ salvataggio in Parquet verificato  